## Lesson Overview

**What this lesson teaches:** how to send local images to Claude, combine an image with a detailed analysis prompt, and process a folder of property images consistently.

**What's happening under the hood:**
1. Read a PNG as bytes and Base64-encode it.
2. Place the image and instructions in one user message.
3. Ask a vision-capable model to assess visible vegetation and roof conditions.
4. Extract visible text from the structured response.
5. Repeat across images while keeping assessments independent.

**You will build:** a reusable workflow for the seven properties in `images/`.

> **Important:** This is an educational visual-screening example, not a professional wildfire inspection. Satellite imagery can be outdated, obstructed, or too low-resolution for definitive safety conclusions.

# Lesson 16: Analyzing Images with Claude

Claude accepts images and text together in a message. Each property image supplies visual evidence and a detailed prompt supplies the assessment criteria. The model returns a short, repeatable fire-risk screening report.

## The Multimodal Request

```text
PNG file → raw bytes → Base64 text ─┐
                                     ├→ user content blocks → Claude → assessment
Assessment instructions ─────────────┘
```

The API needs both the image media type and Base64 data. A helper keeps these details out of the analysis code.

## Setup

Install dependencies once if needed:

```python
%pip install anthropic python-dotenv
```

Add `ANTHROPIC_API_KEY=...` to a `.env` file. Setup checks the current folder and `Claude_API_Training`. Live analysis sends API requests and may incur a charge.

In [1]:
import base64
import os
from pathlib import Path

try:
    from anthropic import Anthropic
except ImportError:
    Anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

for env_path in (Path('.env'), Path('Claude_API_Training/.env')):
    if env_path.exists():
        load_dotenv(env_path)
        break

api_key = os.getenv('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key) if Anthropic is not None and api_key else None
model = 'claude-sonnet-4-5'

if Anthropic is None:
    print('Install anthropic before running the live examples.')
elif client is None:
    print('Add ANTHROPIC_API_KEY to .env before running the live examples.')
else:
    print(f'Claude client ready. Model: {model}')

Claude client ready. Model: claude-sonnet-4-5


## Response Helper

Anthropic responses contain typed content blocks. This helper displays only user-visible `text` blocks instead of assuming every block contains text.

In [2]:
def text_from_message(message):
    return '\n'.join(
        block.text for block in message.content
        if block.type == 'text'
    )

## Define the Assessment Prompt

The prompt separates observations from the final rating and tells the model to acknowledge uncertainty when the image lacks detail. A useful screening tool distinguishes visible evidence from assumptions.

In [3]:
assessment_prompt = """
Analyze the attached satellite image as an educational wildfire-risk screening.
Use only features visible in the image. If image quality, shadows, or
obstructions prevent a reliable observation, say so rather than guessing.

Address each item in one sentence:

1. Residence identification: Locate the likely primary residence using roof
   size, regular geometry, and driveway connections. Describe its position
   relative to other visible structures and property features.

2. Tree overhang: Identify canopy that appears to extend over the roof.
   Estimate apparent roof coverage as 0%, 1-25%, 26-50%, 51-75%, or 76-100%,
   and mention any especially dense areas.

3. Visible fire vulnerabilities: Note apparent canopy-to-roof contact,
   vegetation near roof openings, and vegetation that may bridge surrounding
   fuels to the structure. Do not invent details unsupported by resolution.

4. Defensible space: Describe visible canopy separation, vegetation density
   near the home, and possible fuel ladders.

5. Screening rating:
   - 1 (Low): no visible overhang and good visible separation
   - 2 (Moderate): apparent overhang up to 25% or reduced separation
   - 3 (High): apparent overhang of 26-50%, connected canopies, or several
     visible vulnerabilities
   - 4 (Severe): apparent overhang above 50%, dense vegetation against the
     structure, and limited visible defensible space

End with exactly:
Fire Risk Rating: <1-4> — <brief evidence-based justification>

This rating is a visual screening result, not a professional inspection.
""".strip()

print(assessment_prompt[:500] + '\n...')

Analyze the attached satellite image as an educational wildfire-risk screening.
Use only features visible in the image. If image quality, shadows, or
obstructions prevent a reliable observation, say so rather than guessing.

Address each item in one sentence:

1. Residence identification: Locate the likely primary residence using roof
   size, regular geometry, and driveway connections. Describe its position
   relative to other visible structures and property features.

2. Tree overhang: Identi
...


## Locate and Encode the Images

Base64 converts binary image bytes into text transportable in JSON. The helper validates the extension and returns an Anthropic image source. Paths support launching from either the repository root or `Claude_API_Training`.

In [4]:
image_directory_candidates = (
    Path('images'),
    Path('Claude_API_Training/images'),
)
image_directory = next(
    (path for path in image_directory_candidates if path.exists()),
    None,
)
if image_directory is None:
    raise FileNotFoundError('Could not find the images directory.')

image_paths = sorted(image_directory.glob('prop*.png'))
print(f'Found {len(image_paths)} images:', [p.name for p in image_paths])


def image_source(path):
    path = Path(path)
    media_types = {
        '.png': 'image/png',
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.webp': 'image/webp',
        '.gif': 'image/gif',
    }
    media_type = media_types.get(path.suffix.lower())
    if media_type is None:
        raise ValueError(f'Unsupported image type: {path.suffix}')

    return {
        'type': 'base64',
        'media_type': media_type,
        'data': base64.b64encode(path.read_bytes()).decode('utf-8'),
    }

Found 7 images: ['prop1.png', 'prop2.png', 'prop3.png', 'prop4.png', 'prop5.png', 'prop6.png', 'prop7.png']


## Build a Multimodal User Message

Content blocks are ordered. The image comes first and the instructions follow, making the relationship between evidence and task explicit. This helper returns request data and makes no API call.

In [5]:
def property_analysis_message(path, prompt=assessment_prompt):
    return {
        'role': 'user',
        'content': [
            {'type': 'image', 'source': image_source(path)},
            {'type': 'text', 'text': prompt},
        ],
    }


sample_path = image_paths[0]
sample_message = property_analysis_message(sample_path)
print('Sample:', sample_path.name)
print('Content block types:', [b['type'] for b in sample_message['content']])
print('Encoded characters:', len(sample_message['content'][0]['source']['data']))

Sample: prop1.png
Content block types: ['image', 'text']
Encoded characters: 969064


## Analyze One Property

Start with one image so you can inspect output quality, latency, and cost before scaling up. Without a configured client, the notebook remains runnable through all local sections.

In [6]:
sample_response = None

if client is None:
    print('Skipping live analysis. Complete setup, then rerun this cell.')
elif not image_paths:
    print('No property images found.')
else:
    sample_response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[property_analysis_message(sample_path)],
    )
    print(text_from_message(sample_response))

# Wildfire Risk Screening Analysis

**1. Residence Identification:**
The primary residence appears to be the structure in the center of the image with a light-colored (tan/gray) roof showing regular rectangular geometry, positioned at the intersection of what appears to be a driveway or access path, surrounded by dense vegetation and trees.

**2. Tree Overhang:**
Apparent tree canopy overhang covers approximately 51-75% of the visible roof surface, with particularly dense overhang concentrated on the northern and eastern portions of the structure where dark canopy shadows are clearly visible across the roofline.

**3. Visible Fire Vulnerabilities:**
Multiple tree canopies appear to make direct contact with or extend immediately over the roof surface; vegetation appears to surround the structure on all sides with minimal clearance; dense canopy creates continuous fuel connections from surrounding forest directly to the structure.

**4. Defensible Space:**
There is minimal visible canopy

## Batch the Remaining Properties

Each image uses a separate request, preventing evidence from one property leaking into another assessment. The sample result is reused rather than billed twice.

Seven images means up to seven API calls. Set `run_batch = True` only when you intend to run them.

In [ ]:
run_batch = False
assessments = {}

if sample_response is not None:
    assessments[sample_path.name] = text_from_message(sample_response)

if run_batch and client is None:
    print('Complete the API setup before running the batch.')
elif run_batch:
    for image_path in image_paths:
        if image_path.name in assessments:
            continue
        response = client.messages.create(
            model=model,
            max_tokens=1000,
            messages=[property_analysis_message(image_path)],
        )
        assessments[image_path.name] = text_from_message(response)
        print(f'\n--- {image_path.name} ---')
        print(assessments[image_path.name])
else:
    print('Batch disabled. Set run_batch = True when ready.')
    print('Assessments collected:', list(assessments))

## Save-Friendly Results

Results stay in a dictionary keyed by filename. This is easy to inspect and can later be serialized after human review. The cell previews results without overwriting files.

In [ ]:
for filename, assessment in assessments.items():
    print(f'\n### {filename}')
    print(assessment)

## Local Sanity Checks

These checks make no API calls. They verify the required content blocks, PNG media type, and Base64 round trip.

In [ ]:
assert image_paths, 'Expected at least one property image.'

test_path = image_paths[0]
test_message = property_analysis_message(test_path)
image_block, text_block = test_message['content']

assert test_message['role'] == 'user'
assert image_block['type'] == 'image'
assert image_block['source']['media_type'] == 'image/png'
assert base64.b64decode(image_block['source']['data']) == test_path.read_bytes()
assert text_block['type'] == 'text'
assert 'Fire Risk Rating:' in text_block['text']

print('Local multimodal-message checks passed.')

## Practice: Improve the Image Workflow

Try these one at a time and predict the result:

1. **Change the evidence standard — edit “Define the Assessment Prompt”:** label every claim `visible`, `uncertain`, or `not visible`.
2. **Compare prompts — edit “Analyze One Property”:** run one image with a shorter prompt and compare consistency.
3. **Add structured output — edit the prompt:** request five fixed headings and verify them locally.
4. **Test another format — edit “Locate and Encode the Images”:** add a JPEG and confirm its media type.
5. **Human review — after the batch:** compare each assessment with the image and record disagreements. Never substitute the rating for an on-site inspection.

## Summary

- Multimodal messages combine image and text content blocks.
- Local images must be Base64-encoded and labeled with the correct media type.
- Detailed criteria and uncertainty instructions reduce unsupported claims.
- Test one image before a batch; keep property requests independent.
- Satellite-image output is preliminary screening and requires human or professional validation.

## Extension: Analyze a PDF with the Same Methodology

A PDF follows the same multimodal pattern as an image: read the local file as bytes, Base64-encode those bytes, label the content with the correct media type, and place it beside a text prompt in one user message. The important difference is the content block type: use `document` for a PDF instead of `image`.

Claude can use text, tables, charts, and page imagery contained in the PDF. A precise prompt should identify the task, requested evidence, output structure, and how to handle unreadable or missing information. Page references make the result easier to verify.

```text
PDF file → raw bytes → Base64 text ─┐
                                    ├→ user content blocks → Claude → grounded analysis
Analysis instructions ──────────────┘
```

### Locate and Encode the PDF

The path lookup supports running the notebook from either the repository root or the `Claude_API_Training` directory. The source object mirrors the image helper, but its media type is `application/pdf`.

In [7]:
pdf_candidates = (Path('earth.pdf'), Path('Claude_API_Training/earth.pdf'))
pdf_path = next((path for path in pdf_candidates if path.exists()), None)

if pdf_path is None:
    raise FileNotFoundError('Could not find earth.pdf.')


def pdf_source(path):
    path = Path(path)
    if path.suffix.lower() != '.pdf':
        raise ValueError('Expected a PDF file.')

    return {
        'type': 'base64',
        'media_type': 'application/pdf',
        'data': base64.b64encode(path.read_bytes()).decode('utf-8'),
    }


print(f'PDF ready: {pdf_path.name} ({pdf_path.stat().st_size:,} bytes)')

PDF ready: earth.pdf (886,832 bytes)


### Build the PDF Analysis Message

The document and prompt travel together. This example requests a grounded overview of `earth.pdf`, separates direct evidence from interpretation, and asks for page citations rather than unsupported claims.

In [8]:
pdf_prompt = """
Analyze this PDF and produce a concise, evidence-grounded report.

1. State the document's apparent purpose and intended audience.
2. Summarize its main ideas in 3-5 bullet points.
3. Identify important facts, measurements, tables, charts, or images.
4. Distinguish information stated directly from your interpretation.
5. Note anything unreadable, ambiguous, or absent rather than guessing.

Include page numbers for important claims when the pages are identifiable.
End with three useful follow-up questions that the document could answer.
""".strip()


def pdf_analysis_message(path, prompt=pdf_prompt):
    return {
        'role': 'user',
        'content': [
            {'type': 'document', 'source': pdf_source(path)},
            {'type': 'text', 'text': prompt},
        ],
    }


pdf_message = pdf_analysis_message(pdf_path)
print('Content block types:', [block['type'] for block in pdf_message['content']])

Content block types: ['document', 'text']


### Run the PDF Analysis

The API call is opt-in because it may incur a charge. As with the image workflow, test one document first and inspect whether the answer is grounded in visible content. Set `run_pdf_analysis = True` when ready.

In [10]:
run_pdf_analysis = True
pdf_response = None

if run_pdf_analysis and client is None:
    print('Complete the API setup before analyzing the PDF.')
elif run_pdf_analysis:
    pdf_response = client.messages.create(
        model=model,
        max_tokens=1500,
        messages=[pdf_message],
    )
    print(text_from_message(pdf_response))
else:
    print('PDF analysis disabled. Set run_pdf_analysis = True when ready.')

# Analysis Report: Wikipedia Article on Earth

## 1. Document Purpose and Intended Audience

**Purpose:** This is a Wikipedia encyclopedia article providing comprehensive scientific and factual information about planet Earth.

**Intended Audience:** General public seeking reliable reference information; readers range from students to researchers requiring basic to detailed planetary data.

## 2. Main Ideas

- **Earth as a unique life-supporting planet:** Earth is the third planet from the Sun and the only known astronomical object to harbor life, enabled by its liquid surface water covering 70.8% of its crust (page 1).

- **Physical characteristics:** Earth is an ellipsoid with a circumference of ~40,000 km, mean radius of 6,371 km, mass of 5.972 × 10²⁴ kg, and is the densest planet in the Solar System (pages 1-2).

- **Dynamic systems:** Earth has a dynamic atmosphere (78% nitrogen, 21% oxygen), tectonic plates, liquid outer core generating a magnetosphere, and a tilted axis producing

### Use API Citations with PDF or Plain Text

Asking for page numbers in a prompt relies on the model to format them correctly. Anthropic's citations feature instead attaches structured source locations to response text. Enable it with `citations: {'enabled': True}` on a `document` content block. PDF citations identify page ranges; plain-text citations identify character ranges. Citation indices are zero-based, so add 1 when showing a PDF page number to a reader.

Citations work with document blocks, not ordinary user `text` blocks. For `.txt`, Markdown, or other text you have already extracted, send the contents as a text-backed `document`.

In [11]:
cited_pdf_message = {
    'role': 'user',
    'content': [
        {
            'type': 'document',
            'source': pdf_source(pdf_path),
            'title': pdf_path.name,
            'citations': {'enabled': True},
        },
        {
            'type': 'text',
            'text': 'What are the three most important claims? Support each claim with citations.',
        },
    ],
}

# For .txt, .md, or extracted text, use a text-backed document instead.
plain_text = "Earth is the third planet from the Sun. It has one natural satellite."
cited_text_message = {
    'role': 'user',
    'content': [
        {
            'type': 'document',
            'source': {
                'type': 'text',
                'media_type': 'text/plain',
                'data': plain_text,
            },
            'title': 'earth-notes.txt',
            'citations': {'enabled': True},
        },
        {'type': 'text', 'text': 'Summarize the facts and cite the source.'},
    ],
}

print('PDF citations enabled:', cited_pdf_message['content'][0]['citations']['enabled'])
print('Text citations enabled:', cited_text_message['content'][0]['citations']['enabled'])

PDF citations enabled: True
Text citations enabled: True


The response still contains text blocks, but a cited block also has a `citations` list. Each entry includes the quoted source text and its location. This opt-in call prints both the answer and the structured metadata so you can render citations in a UI or verify them programmatically.

In [12]:
run_citation_example = True
citation_response = None

if run_citation_example and client is None:
    print('Complete the API setup before running the citation example.')
elif run_citation_example:
    citation_response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=[cited_pdf_message],  # Swap in cited_text_message for plain text.
    )

    for block in citation_response.content:
        if block.type == 'text':
            print(block.text)
            for citation in block.citations or []:
                print('  citation:', citation.model_dump())
else:
    print('Citation example disabled. Set run_citation_example = True when ready.')

Based on the document, here are three of the most important claims about Earth:

1. **Earth is the only known planet that harbors life**: 
Earth is the third planet from the Sun and the only astronomical object known to harbor life.
  citation: {'cited_text': "Earth\r\nThe Blue Marble, Apollo 17, December 1972\r\nDesignations\r\nAlternative\r\nnames\r\nThe world · The globe ·\r\nTerra · Tellus · Gaia ·\r\nMother Earth · Sol III\r\nAdjectives Earthly · Terrestrial · Terran\r\n· Tellurian\r\nSymbol and\r\nOrbital characteristics\r\nEpoch J2000\r\n[n 1]\r\nAphelion 152 097 597 km\r\nPerihelion 147 098 450 km\r\n[n 2]\r\nSemi-major axis 149 598 023 km\r\n[1]\r\nEccentricity 0.016 7086\r\n[1]\r\nOrbital period\r\n(sidereal)\r\n365.256 363 004 d\r\n[2]\r\n(1.000 017 420 96 aj)\r\nAverage orbital\r\nspeed\r\n29.7827 km/s\r\n[3]\r\nMean anomaly 358.617°\r\nInclination 7.155° – Sun's equator;\r\nEarth\r\nEarth is the third planet from the Sun and the only\r\nastronomical object known to harbor 

### Local PDF Sanity Check

This check makes no API call. It confirms that the PDF uses a document block, has the correct media type, and survives the Base64 round trip without changing its bytes.

In [ ]:
document_block, pdf_text_block = pdf_message['content']

assert document_block['type'] == 'document'
assert document_block['source']['media_type'] == 'application/pdf'
assert base64.b64decode(document_block['source']['data']) == pdf_path.read_bytes()
assert pdf_text_block['type'] == 'text'
assert 'page numbers' in pdf_text_block['text'].lower()

print('Local PDF-message checks passed.')